# LeetCode 2566 — Maximum Difference by Remapping a Digit bằng Python

Link đề: https://leetcode.com/problems/maximum-difference-by-remapping-a-digit/ · Độ khó: **Easy** · Chủ đề: Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minMaxDifference(self, num: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu phép **"ánh xạ một chữ số"**: chọn chữ số `d1` và chữ số `d2`, thay **mọi** chữ số `d1` trong `num` bằng `d2`; kết quả có thể có chữ số `0` ở đầu.
2. Biết rằng đáp án là `(giá trị lớn nhất) - (giá trị nhỏ nhất)` có thể có được, và hai giá trị đó được chọn **độc lập** với nhau.
3. Nắm chiến lược **tham lam (greedy)**: để lớn nhất, đổi chữ số **không phải 9** đầu tiên (từ trái) thành `9`; để nhỏ nhất, đổi chữ số **đầu tiên** thành `0`.
4. Viết 4 cách: thử cả `10 x 10` phép ánh xạ, tham lam với `str.replace`, tham lam với bảng dịch `translate`, và công thức theo **giá trị vị trí** (place value); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: chỉ thay một lần xuất hiện, quên nhỏ nhất, thay chữ số sai cho giá trị nhỏ nhất, và bắt đầu tìm chữ số không phải 9 từ bên phải.

## 2. Đề bài & ví dụ

Cho số nguyên `num`. Bạn **được** chọn đúng **hai** chữ số `d1` và `d2` (từ `0` đến `9`, có thể bằng nhau) rồi **thay mọi** chữ số `d1` trong `num` bằng `d2` (gọi là *ánh xạ* `d1 -> d2`). Bạn thực hiện việc này **hai lần độc lập** trên `num` ban đầu để được:

- `max_val`: giá trị **lớn nhất** có thể có;
- `min_val`: giá trị **nhỏ nhất** có thể có.

Trả về `max_val - min_val`. Lưu ý: kết quả sau khi ánh xạ **có thể có số `0` ở đầu** (khi đó là giá trị nhỏ hơn bình thường, ví dụ `"007"` là `7`).

| Ví dụ | num | Output | Giải thích |
|---|---|---|---|
| 1 | `11891` | `99009` | lớn nhất `99899` (`1 -> 9`), nhỏ nhất `890` (`1 -> 0`, được `00890`); hiệu `99009` |
| 2 | `90` | `99` | lớn nhất `99` (`0 -> 9`), nhỏ nhất `0` (`9 -> 0`, được `00`); hiệu `99` |

Constraints: `1 <= num <= 10^8`.

In [1]:
examples = [(11891, 99009), (90, 99)]
for num, expected in examples:
    s = str(num)
    values = sorted({int(s.replace(a, b)) for a in "0123456789" for b in "0123456789"})
    print(f"num={num}: nhỏ nhất={values[0]}, lớn nhất={values[-1]} -> hiệu {values[-1] - values[0]} (mong đợi {expected})")

num=11891: nhỏ nhất=890, lớn nhất=99899 -> hiệu 99009 (mong đợi 99009)
num=90: nhỏ nhất=0, lớn nhất=99 -> hiệu 99 (mong đợi 99)


## 3. Kiến thức nền

### 3.1. Chuỗi số và `str.replace`

`str(num)` đổi số sang chuỗi chữ số. `s.replace(a, b)` thay **mọi** lần xuất hiện của `a` bằng `b` và trả chuỗi mới (không đổi `s`). Có thể thêm đối số thứ ba `count` để giới hạn số lần thay.

In [2]:
s = "11891"
print(s.replace("1", "9"))
print(s.replace("1", "9", 1))      # chỉ thay lần xuất hiện đầu tiên
print(s)                           # s không đổi

99899
91891
11891


### 3.2. `int` với chuỗi có số `0` ở đầu

`int("00890")` cho `890`: chữ số `0` ở đầu bị bỏ khi chuyển thành số, nên kết quả hợp lệ để so sánh bằng giá trị số.

In [3]:
print(int("00890"), int("00"), int("0099"))

890 0 99


### 3.3. `str.translate` và `str.maketrans`

`str.maketrans(a, b)` tạo bảng dịch thay ký tự `a` bằng `b` (khi `a`, `b` là chuỗi một ký tự). `s.translate(bảng)` áp dụng bảng cho toàn chuỗi trong một lượt.

In [4]:
table = str.maketrans("1", "9")
print("11891".translate(table))
table0 = str.maketrans("1", "0")
print("11891".translate(table0), int("11891".translate(table0)))

99899
00890 890


### 3.4. Giá trị vị trí (place value)

Chữ số ở vị trí thứ `p` tính từ phải (bắt đầu từ `0`) đóng góp `digit * 10^p`. Ví dụ `11891 = 1*10^4 + 1*10^3 + 8*10^2 + 9*10^1 + 1*10^0`. Nếu thay **mọi** chữ số `d` bằng `e`, số thay đổi `(e - d) * P(d)` với `P(d)` là **tổng `10^p` của các vị trí có chữ số `d`**.

In [5]:
s = "11891"
n = len(s)
P = {}
for idx, ch in enumerate(s):
    p = n - 1 - idx
    P[ch] = P.get(ch, 0) + 10 ** p
print(P)                                   # '1': 10000 + 1000 + 1 = 11001
print(11891 + (9 - 1) * P["1"])            # thay 1 -> 9

{'1': 11001, '8': 100, '9': 10}
99899


### 3.5. `set` và biểu thức tập hợp

`{int(s.replace(a, b)) for a in digits for b in digits}` dựng **tập** các giá trị khác nhau có thể có (có thể trùng nhau ở nhiều cặp `(a, b)`). Sau đó dùng `max(...)`/`min(...)`.

In [6]:
s = "90"
vals = {int(s.replace(a, b)) for a in "0123456789" for b in "0123456789"}
print(sorted(vals))
print(max(vals), min(vals))

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
99 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `num = 11891`:

- **Lớn nhất:** ta muốn các chữ số cao nhất có thể, ưu tiên chữ số **bên trái** (giá trị vị trí lớn nhất). Chữ số trái nhất là `1`, chưa phải `9`; đổi **mọi** `1` thành `9` được `99899`. Không có ánh xạ nào cho giá trị lớn hơn vì chữ số đầu đã là `9`, và chữ số thứ hai cũng đã `9` rồi.
- **Nhỏ nhất:** ta muốn các chữ số cao nhất **nhỏ nhất có thể**: đổi chữ số đầu tiên (`1`) thành `0`; mọi `1` thành `0` được `00890 = 890`.
- Hiệu: `99899 - 890 = 99009`.

Với `num = 90`:

- Lớn nhất: chữ số đầu là `9` (không đổi); chữ số không phải 9 đầu tiên là `0`; đổi `0 -> 9` được `99`.
- Nhỏ nhất: đổi chữ số đầu `9 -> 0` được `00 = 0`.
- Hiệu: `99 - 0 = 99`.

In [7]:
for num in (11891, 90, 5, 999, 123):
    s = str(num)
    a = next((c for c in s if c != "9"), None)        # chữ số không phải 9 đầu tiên
    big = int(s.replace(a, "9")) if a else num
    small = int(s.replace(s[0], "0"))
    print(f"num={num}: lớn nhất={big} (đổi {a} -> 9), nhỏ nhất={small} (đổi {s[0]} -> 0), hiệu={big - small}")

num=11891: lớn nhất=99899 (đổi 1 -> 9), nhỏ nhất=890 (đổi 1 -> 0), hiệu=99009
num=90: lớn nhất=99 (đổi 0 -> 9), nhỏ nhất=0 (đổi 9 -> 0), hiệu=99
num=5: lớn nhất=9 (đổi 5 -> 9), nhỏ nhất=0 (đổi 5 -> 0), hiệu=9
num=999: lớn nhất=999 (đổi None -> 9), nhỏ nhất=0 (đổi 9 -> 0), hiệu=999
num=123: lớn nhất=923 (đổi 1 -> 9), nhỏ nhất=23 (đổi 1 -> 0), hiệu=900


## 5. Cách 1 — Brute Force / baseline: thử cả `10 x 10` ánh xạ

**Ý tưởng:** với mọi cặp `(a, b)` trong `0..9`, tính `int(s.replace(a, b))`. Lấy `max` và `min` của tất cả giá trị rồi trừ nhau. Chỉ có `100` ánh xạ nên hoàn toàn khả thi.

In [8]:
class SolutionBrute:
    def minMaxDifference(self, num: int) -> int:
        s = str(num)
        digits = "0123456789"
        values = [int(s.replace(a, b)) for a in digits for b in digits]
        return max(values) - min(values)

In [9]:
sol = SolutionBrute()
print(sol.minMaxDifference(11891))    # 99009
print(sol.minMaxDifference(90))       # 99

99009
99


**Độ phức tạp:** Time `O(100 * L)` với `L` là số chữ số của `num` (`L <= 9`), Space `O(L)`. Về lý thuyết là hằng số. **Nút thắt:** không có; chỉ là cách làm "thử hết", nhưng ta có thể chỉ rõ ánh xạ tối ưu bằng suy luận tham lam để hiểu bản chất.

## 6. Ý tưởng thuật toán / bước đột phá

Hai giá trị cần tìm **độc lập**:

- **`max_val`:** quét chữ số từ trái sang phải; chữ số **đầu tiên khác 9** là `a`; thay mọi `a` bằng `9`. Nếu mọi chữ số đều là `9` thì số đã lớn nhất, không đổi.
  - Lý do: thay `a -> 9` tăng giá trị thêm `(9 - a) * P(a)`; ưu tiên chữ số cao nhất khác 9 vì nó chứa `10^p` lớn nhất trong `P(a)`.
- **`min_val`:** thay mọi chữ số **đầu tiên** `s[0]` bằng `0`.
  - Lý do: giảm `s[0] * P(s[0])`; chữ số đầu tiên có giá trị vị trí lớn nhất nên giảm nó về `0` cho mức giảm lớn nhất (với `s[0] >= 1` luôn giảm; `num >= 1` nên `s[0] >= 1`).

**Quy tắc:** *`max`: đổi chữ số không-phải-9 đầu tiên thành 9; `min`: đổi chữ số đầu thành 0; đáp án `max - min`.*

**Invariant:** ánh xạ chọn được không thể bị cải thiện bằng cách đổi sang chữ số khác vì nếu một chữ số cao hơn chưa tối ưu (chưa là 9 cho max, chưa là 0 cho min) thì có thể cải thiện ở vị trí đó với đóng góp lớn hơn mọi vị trí thấp hơn.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam với `str.replace`

In [10]:
class SolutionGreedy:
    def minMaxDifference(self, num: int) -> int:
        s = str(num)
        # lớn nhất: đổi chữ số không phải 9 đầu tiên thành 9
        big = num
        for ch in s:
            if ch != "9":
                big = int(s.replace(ch, "9"))
                break
        # nhỏ nhất: đổi chữ số đầu thành 0
        small = int(s.replace(s[0], "0"))
        return big - small

In [11]:
sol = SolutionGreedy()
print(sol.minMaxDifference(11891))    # 99009
print(sol.minMaxDifference(90))       # 99

99009
99


**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

### Cách 3 — Tham lam với bảng dịch `translate`

Giống Cách 2 nhưng dùng `str.maketrans` + `translate` để mô tả rõ ánh xạ `d1 -> d2`.

In [12]:
class SolutionTranslate:
    def minMaxDifference(self, num: int) -> int:
        s = str(num)
        non9 = next((ch for ch in s if ch != "9"), None)
        to_max = str.maketrans(non9, "9") if non9 is not None else str.maketrans("", "")   # bảng rỗng: không đổi
        to_min = str.maketrans(s[0], "0")
        return int(s.translate(to_max)) - int(s.translate(to_min))

In [13]:
sol = SolutionTranslate()
print(sol.minMaxDifference(11891))    # 99009
print(sol.minMaxDifference(90))       # 99
print(sol.minMaxDifference(999))      # 999

99009
99
999


**Độ phức tạp:** Time `O(L)`, Space `O(L)`. `str.maketrans("", "")` tạo bảng rỗng (không đổi gì), dùng khi mọi chữ số đã là `9`.

### Cách 4 — Công thức theo giá trị vị trí (place value)

Không cần chuỗi mới: tính `P(d)` = tổng `10^p` của các vị trí có chữ số `d`. Khi đó:

- `max_val = num + (9 - a) * P(a)` với `a` là chữ số không-phải-9 trái nhất;
- `min_val = num - b * P(b)` với `b` là chữ số đầu.

Vậy `đáp án = (9 - a) * P(a) + b * P(b)` (số hạng đầu bằng `0` nếu không có `a`).

In [14]:
class SolutionPlaceValue:
    def minMaxDifference(self, num: int) -> int:
        digits = []
        t = num
        while t > 0:
            t, d = divmod(t, 10)
            digits.append(d)
        digits.reverse()                         # digits[0] là chữ số đầu
        L = len(digits)
        P = [0] * 10                             # P[d] = tổng 10^p của các vị trí có chữ số d
        for idx, d in enumerate(digits):
            P[d] += 10 ** (L - 1 - idx)
        gain = 0
        for d in digits:
            if d != 9:
                gain = (9 - d) * P[d]            # tăng khi đổi d -> 9
                break
        loss = digits[0] * P[digits[0]]          # giảm khi đổi chữ số đầu -> 0
        return gain + loss

In [15]:
sol = SolutionPlaceValue()
print(sol.minMaxDifference(11891))    # 99009
print(sol.minMaxDifference(90))       # 99
print(sol.minMaxDifference(999))      # 999

99009
99
999


**Độ phức tạp:** Time `O(L)`, Space `O(1)` (mảng `P` 10 ô và danh sách chữ số `<= 9`). Cách này thể hiện rõ vì sao greedy đúng: mỗi ánh xạ là một thay đổi `(e - d) * P(d)` có thể tính trực tiếp.

## 8. TRACE từng bước

Trace Cách 4 với `num = 11891` (chữ số `1,1,8,9,1`; độ dài 5):

| Bước | Nội dung | Kết quả |
|---|---|---|
| 1 | `P[1]` = `10^4 + 10^3 + 10^0` | `11001` |
| 2 | `P[8] = 10^2`, `P[9] = 10^1` | `100`, `10` |
| 3 | chữ số không-phải-9 đầu tiên `d = 1` | `gain = (9 - 1) * 11001 = 88008` |
| 4 | chữ số đầu `b = 1` | `loss = 1 * 11001 = 11001` |
| 5 | đáp án | `88008 + 11001 = 99009` |

Kiểm tra: `max = 11891 + 88008 = 99899`, `min = 11891 - 11001 = 890`, hiệu `99899 - 890 = 99009`.

In [16]:
def trace_place_value(num):
    digits = [int(c) for c in str(num)]
    L = len(digits)
    P = [0] * 10
    for idx, d in enumerate(digits):
        P[d] += 10 ** (L - 1 - idx)
    print("num =", num, "| chữ số:", digits)
    print("  P (khác 0):", {d: P[d] for d in range(10) if P[d]})
    gain = 0
    for d in digits:
        if d != 9:
            gain = (9 - d) * P[d]
            print(f"  chữ số không phải 9 đầu tiên: {d} -> gain = (9 - {d}) * {P[d]} = {gain}")
            break
    else:
        print("  mọi chữ số đều là 9 -> gain = 0")
    b = digits[0]
    loss = b * P[b]
    print(f"  chữ số đầu: {b} -> loss = {b} * {P[b]} = {loss}")
    print(f"  max = {num + gain}, min = {num - loss}, hiệu = {gain + loss}")
    return gain + loss

trace_place_value(11891)
trace_place_value(90)
trace_place_value(999)
trace_place_value(5)

num = 11891 | chữ số: [1, 1, 8, 9, 1]
  P (khác 0): {1: 11001, 8: 100, 9: 10}
  chữ số không phải 9 đầu tiên: 1 -> gain = (9 - 1) * 11001 = 88008
  chữ số đầu: 1 -> loss = 1 * 11001 = 11001
  max = 99899, min = 890, hiệu = 99009
num = 90 | chữ số: [9, 0]
  P (khác 0): {0: 1, 9: 10}
  chữ số không phải 9 đầu tiên: 0 -> gain = (9 - 0) * 1 = 9
  chữ số đầu: 9 -> loss = 9 * 10 = 90
  max = 99, min = 0, hiệu = 99
num = 999 | chữ số: [9, 9, 9]
  P (khác 0): {9: 111}
  mọi chữ số đều là 9 -> gain = 0
  chữ số đầu: 9 -> loss = 9 * 111 = 999
  max = 999, min = 0, hiệu = 999
num = 5 | chữ số: [5]
  P (khác 0): {5: 1}
  chữ số không phải 9 đầu tiên: 5 -> gain = (9 - 5) * 1 = 4
  chữ số đầu: 5 -> loss = 5 * 1 = 5
  max = 9, min = 0, hiệu = 9


9

## 9. Code nộp LeetCode

Tham lam với `str.replace`:

In [17]:
class Solution:
    def minMaxDifference(self, num: int) -> int:
        s = str(num)
        big = num
        for ch in s:
            if ch != "9":
                big = int(s.replace(ch, "9"))
                break
        small = int(s.replace(s[0], "0"))
        return big - small

In [18]:
print(Solution().minMaxDifference(11891))
print(Solution().minMaxDifference(90))

99009
99


## 10. Vì sao thuật toán đúng

1. **Hai bài toán độc lập.** Giá trị lớn nhất và nhỏ nhất đều lấy từ cùng tập `100` ánh xạ trên `num` ban đầu, nên tối ưu từng cái riêng rẽ rồi trừ là đúng.
2. **Thay đổi do một ánh xạ.** Ánh xạ `d -> e` thay đổi số một lượng `(e - d) * P(d)` với `P(d) >= 0` là tổng `10^p` của các vị trí chứa `d`. Với `d` không xuất hiện, `P(d) = 0` nên không đổi.
3. **Lớn nhất.** Gọi `a` là chữ số không-phải-9 **trái nhất**, đứng ở vị trí `p` (tính từ phải). Đổi `a -> 9` tăng ít nhất `(9 - a) * 10^p >= 10^p`. Mọi chữ số bên trái `a` đều là `9` nên đổi chúng chỉ làm số giảm hoặc giữ nguyên. Mọi chữ số khác `a` (không phải 9) chỉ nằm ở vị trí `< p`, nên đổi chúng tăng tối đa `9 * (10^(p-1) + ... + 1) < 10^p`, tức là kém hơn cách đổi `a`. Vậy `a -> 9` là tối ưu.
4. **Nhỏ nhất.** Mức giảm `d * P(d)` với `e = 0` là lớn nhất cho mỗi `d`. Chữ số đầu `b = s[0] >= 1` có đóng góp vị trí cao nhất `b * 10^(L-1)`, lớn hơn bất kỳ chữ số nào khác không thể vượt quá; chọn nó để giảm là tốt nhất (và kết quả có thể có `0` ở đầu, hợp lệ theo đề).
5. **Trường hợp mọi chữ số là 9.** Số đã lớn nhất; tăng thêm bằng `0`. Đáp án `= num - min = num - 0 = num` (vì mọi chữ số `9 -> 0` ra `0`).
6. **Các cách khác tương đương.** Cách `translate` và `place value` mô tả cùng ánh xạ; Cách brute thử hết nên là chuẩn để so sánh.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử `10 x 10` ánh xạ | `O(100 * L)` | `O(L)` |
| 2. Tham lam `replace` | `O(L)` | `O(L)` |
| 3. Tham lam `translate` | `O(L)` | `O(L)` |
| 4. Place value | `O(L)` | `O(1)` |

`L = len(str(num)) <= 9` (vì `num <= 10^8`). Mọi cách thực tế là hằng số.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Các kỳ vọng được tính bằng một vòng thử `10 x 10` ánh xạ độc lập (ghi chú trong cell).

In [19]:
sols = [SolutionBrute(), SolutionGreedy(), SolutionTranslate(), SolutionPlaceValue(), Solution()]
cases = [
    (11891, 99009),        # ví dụ 1
    (90, 99),              # ví dụ 2
    (1, 9),                # nhỏ nhất: max 9, min 0
    (5, 9),                # một chữ số: 9 - 0
    (9, 9),                # một chữ số 9: max 9, min 0
    (10, 90),              # max 90 (1 -> 9; còn 0 -> 9 chỉ được 19), min 0 (1 -> 0 cho 00)
    (19, 90),              # max 99 (1 -> 9), min 9 (1 -> 0 cho 09)
    (91, 98),              # max 99, min 1 (9 -> 0 cho 01)
    (99, 99),              # toàn 9: max 99, min 0
    (55, 99),              # max 99, min 0
    (100, 900),            # max 900 (1 -> 9; còn 0 -> 9 chỉ được 199), min 0 (1 -> 0 cho 000)
    (999, 999),            # toàn 9
    (123456, 900000),
    (908, 990),
    (909, 999),
    (12121, 90909),
    (70007, 90009),
    (98765432, 91000000),
    (10101010, 90909090),
    (99999999, 99999999),  # tám chữ số 9
    (100000000, 900000000),    # biên trên 10^8
]
for num, want in cases:
    for s in sols:
        got = s.minMaxDifference(num)
        assert got == want, (type(s).__name__, num, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 21


Kiểm tra tay vài ca: `num = 100`: lớn nhất đổi `1 -> 9` được `900` (đổi `0 -> 9` chỉ được `199`); nhỏ nhất đổi `1 -> 0` được `000 = 0`; hiệu `900`. `num = 19`: lớn nhất `99`, nhỏ nhất `09 = 9`; hiệu `90`. `num = 91`: lớn nhất `99`, nhỏ nhất `01 = 1`; hiệu `98`.

### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `num` đúng constraints `1 <= num <= 10^8` nhiều kiểu: chọn số chữ số `1..9` rồi chữ số ngẫu nhiên (chữ số đầu khác `0`), kèm số có nhiều chữ số lặp, nhiều số `9`, nhiều số `0`.

In [20]:
import random
random.seed(2566)

def gen():
    kind = random.randrange(5)
    L = random.randint(1, 8)
    if kind == 0:
        digs = [random.choice("0123456789") for _ in range(L)]
    elif kind == 1:
        digs = [random.choice("19") for _ in range(L)]
    elif kind == 2:
        digs = [random.choice("90") for _ in range(L)]
    elif kind == 3:
        digs = [random.choice("123") for _ in range(L)]
    else:
        digs = ["9"] * L
    if digs[0] == "0":
        digs[0] = random.choice("123456789")
    num = int("".join(digs))
    return num

def oracle(num):
    s = str(num)
    vals = [int(s.replace(a, b)) for a in "0123456789" for b in "0123456789"]
    return max(vals) - min(vals)

for _ in range(4000):
    num = gen()
    assert 1 <= num <= 10**8
    ans = [s.minMaxDifference(num) for s in sols]
    assert len(set(ans)) == 1 and ans[0] == oracle(num), (num, ans)
print("4000 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle")

4000 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle


### 12.3. Quét toàn bộ số nhỏ và vài biên

Duyệt mọi `num` từ `1` đến `20000` và so sánh các cách tham lam với brute; thêm biên `10^8` và `99999999`.

In [21]:
for num in range(1, 20001):
    r = SolutionBrute().minMaxDifference(num)
    assert SolutionGreedy().minMaxDifference(num) == r, num
    assert SolutionPlaceValue().minMaxDifference(num) == r, num
assert Solution().minMaxDifference(10**8) == 900000000
assert Solution().minMaxDifference(99999999) == 99999999
print("đã kiểm tra toàn bộ 1..20000 và các biên")

đã kiểm tra toàn bộ 1..20000 và các biên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ thay lần xuất hiện **đầu tiên** của chữ số

In [22]:
def sai_mot_lan(num):
    s = str(num)
    big = num
    for ch in s:
        if ch != "9":
            big = int(s.replace(ch, "9", 1))           # SAI: chỉ thay một lần
            break
    small = int(s.replace(s[0], "0"))
    return big - small

print("đúng:", Solution().minMaxDifference(11891), "| sai:", sai_mot_lan(11891))
assert Solution().minMaxDifference(11891) != sai_mot_lan(11891)

đúng: 99009 | sai: 91001


**Vì sao sai:** phép ánh xạ thay **mọi** lần xuất hiện của chữ số, không chỉ một lần. Thay một lần cho `91891` (nhỏ hơn `99899`).

### Lỗi 2: Quên giá trị nhỏ nhất (lấy `num` làm nhỏ nhất)

In [23]:
def sai_quen_min(num):
    s = str(num)
    big = num
    for ch in s:
        if ch != "9":
            big = int(s.replace(ch, "9"))
            break
    return big - num                                   # SAI: nhỏ nhất cũng có thể giảm bằng ánh xạ

print("đúng:", Solution().minMaxDifference(90), "| sai:", sai_quen_min(90))
assert Solution().minMaxDifference(90) != sai_quen_min(90)

đúng: 99 | sai: 9


**Vì sao sai:** đề cho phép ánh xạ để làm giá trị **nhỏ hơn**. Đổi chữ số đầu thành `0` luôn giảm số (vì `num >= 1`).

### Lỗi 3: Đổi chữ số **nhỏ nhất** thành `0` thay vì chữ số **đầu**

In [24]:
def sai_chu_so_nho_nhat(num):
    s = str(num)
    big = num
    for ch in s:
        if ch != "9":
            big = int(s.replace(ch, "9"))
            break
    small = int(s.replace(min(s), "0"))                # SAI: chữ số nhỏ nhất, không phải chữ số đầu
    return big - small

print("đúng:", Solution().minMaxDifference(90), "| sai:", sai_chu_so_nho_nhat(90))
assert Solution().minMaxDifference(90) != sai_chu_so_nho_nhat(90)

đúng: 99 | sai: 9


**Vì sao sai:** muốn số **nhỏ nhất**, hãy giảm chữ số ở vị trí cao nhất (chữ số đầu) về `0`. Với `90`, chữ số nhỏ nhất là `0` (đổi `0 -> 0` không đổi gì), trong khi đổi `9 -> 0` cho `0`.

### Lỗi 4: Tìm chữ số không phải 9 từ bên **phải**

In [25]:
def sai_tu_phai(num):
    s = str(num)
    big = num
    for ch in reversed(s):                             # SAI: phải ưu tiên chữ số bên trái
        if ch != "9":
            big = int(s.replace(ch, "9"))
            break
    small = int(s.replace(s[0], "0"))
    return big - small

print("đúng:", Solution().minMaxDifference(81), "| sai:", sai_tu_phai(81))
assert Solution().minMaxDifference(81) != sai_tu_phai(81)

đúng: 90 | sai: 88


**Vì sao sai:** chữ số bên trái có giá trị vị trí lớn hơn. Với `81`: đổi `8 -> 9` được `91`, còn đổi `1 -> 9` chỉ được `89`.

### Lỗi 5: Đổi chữ số đầu thành `9` mà không kiểm tra nó đã là `9` chưa

In [26]:
def sai_chu_so_dau(num):
    s = str(num)
    big = int(s.replace(s[0], "9"))                    # SAI: nếu s[0] đã là 9 thì không đổi gì
    small = int(s.replace(s[0], "0"))
    return big - small

print("đúng:", Solution().minMaxDifference(90), "| sai:", sai_chu_so_dau(90))
assert Solution().minMaxDifference(90) != sai_chu_so_dau(90)

đúng: 99 | sai: 90


**Vì sao sai:** khi chữ số đầu đã là `9` thì cần chuyển sang chữ số không-phải-9 **kế tiếp** (như `0` trong `90`). Với `90` cách sai ra `90 - 0 = 90`, đúng là `99`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "đổi chữ số để lớn nhất / nhỏ nhất" ⇒ **tham lam theo vị trí từ trái sang phải**: chữ số trái có trọng số lớn nhất.
- Muốn lớn nhất: đổi chữ số không-phải-9 đầu tiên thành `9`. Muốn nhỏ nhất: đổi chữ số đầu thành `0` (đề cho phép số `0` ở đầu).
- Chỉ có `100` ánh xạ nên luôn có thể **brute force để kiểm tra** greedy.
- Ánh xạ thay **mọi** lần xuất hiện: dùng `str.replace` (không giới hạn `count`).
- `int("00890") = 890`, nên so sánh bằng giá trị số không lo số `0` ở đầu.
- Nhớ trường hợp mọi chữ số là `9` (không có chữ số để tăng).

## 15. Biến thể của bài

1. **LeetCode 1432 "Max Difference You Can Get From Changing an Integer":** chọn `x`, `y`, thay mọi `x` bằng `y`, kết quả **không** được có số `0` ở đầu và **không** là `0`.
2. **LeetCode 1323 "Maximum 69 Number":** chỉ được đổi **một** chữ số `6 -> 9`.
3. **Cho phép đổi tối đa `k` ánh xạ** (nhiều lần ánh xạ liên tiếp).
4. **Số lượng giá trị khác nhau** tạo ra khi thử mọi ánh xạ `d1 -> d2`.
5. **Số nhỏ nhất không có số `0` ở đầu:** đổi chữ số đầu thành `1`, các chữ số còn lại khác chữ số đầu thành `0`.
6. **Dạng chuỗi dài (đến `10^5` chữ số):** vẫn tham lam `O(L)`, brute `O(100 L)` vẫn được.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `10 x 10` ánh xạ | `replace` mọi cặp `(a, b)` | `O(100 L)` | `O(L)` | oracle, kiểm tra greedy |
| 2. Tham lam `replace` | đổi chữ số không-phải-9 đầu tiên; đổi chữ số đầu thành 0 | `O(L)` | `O(L)` | cách nộp bài chuẩn |
| 3. Tham lam `translate` | như Cách 2, dùng bảng dịch | `O(L)` | `O(L)` | khi muốn mô tả ánh xạ rõ ràng |
| 4. Place value | `(9 - a) * P(a) + b * P(b)` | `O(L)` | `O(1)` | khi tránh tạo chuỗi, hiểu bản chất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().minMaxDifference(5)` bằng bao nhiêu? Còn `Solution().minMaxDifference(100)`?

**Bài 2 (tính tay).** Với `num = 2736`, tìm giá trị lớn nhất, nhỏ nhất và hiệu.

**Bài 3 (điền TODO).** Hoàn thành `max_after_remap(num)` trả về **chỉ** giá trị lớn nhất có thể có.

In [27]:
def max_after_remap(num):
    s = str(num)
    # TODO: tìm chữ số đầu tiên khác '9', thay mọi lần xuất hiện của nó bằng '9'; nếu không có thì trả num
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `min_no_leading_zero(num)` (LeetCode 1432): giá trị nhỏ nhất sau khi thay mọi chữ số `x` bằng `y` sao cho kết quả **không có số `0` ở đầu** và **khác `0`**.

**Bài 5 (biến thể / mini challenge).** Viết `count_distinct_results(num)` đếm số **giá trị khác nhau** có thể có khi thử mọi ánh xạ `d1 -> d2` (với `d1, d2` từ `0` đến `9`). Kiểm tra `count_distinct_results(5) == 10` (các giá trị `0..9` từ ánh xạ `5 -> b`; giá trị `5` khi không đổi đã nằm trong đó) và `count_distinct_results(12) == 19`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5`: lớn nhất `9`, nhỏ nhất `0`, hiệu `9`. `100`: lớn nhất `900`, nhỏ nhất `0`, hiệu `900`.
- **Mức 2 (vừa):** Bài 2: `2736`: đổi `2 -> 9` được `9736`; đổi `2 -> 0` được `0736 = 736`; hiệu `9000`. Bài 3: vòng `for ch in s: if ch != '9': return int(s.replace(ch, '9'))`.
- **Mức 3 (sâu):** Bài 4: nếu `s[0] != '1'` thì đổi `s[0] -> '1'`; ngược lại tìm chữ số khác `0` và khác `'1'` (không phải chữ số đầu) rồi đổi thành `0`. Bài 5: dùng tập `{int(s.replace(a, b)) ...}` rồi đếm; chú ý các ánh xạ có `a` không xuất hiện đều cho `num`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert Solution().minMaxDifference(5) == 9
assert Solution().minMaxDifference(100) == 900

# Bài 2
assert Solution().minMaxDifference(2736) == 9000

# Bài 3
def max_after_remap(num):
    s = str(num)
    for ch in s:
        if ch != "9":
            return int(s.replace(ch, "9"))
    return num

# Bài 4 (LeetCode 1432: nhỏ nhất, không có số 0 ở đầu, khác 0)
def min_no_leading_zero(num):
    s = str(num)
    if s[0] != "1":
        return int(s.replace(s[0], "1"))
    for ch in s[1:]:
        if ch not in "01":
            return int(s.replace(ch, "0"))
    return num

# Bài 5
def count_distinct_results(num):
    s = str(num)
    return len({int(s.replace(a, b)) for a in "0123456789" for b in "0123456789"})

assert max_after_remap(11891) == 99899
assert max_after_remap(999) == 999
assert max_after_remap(90) == 99
assert min_no_leading_zero(555) == 111           # LeetCode 1432 ví dụ 1: min là 111
assert min_no_leading_zero(9) == 1               # LeetCode 1432 ví dụ 2: min là 1
assert min_no_leading_zero(1000) == 1000
assert min_no_leading_zero(123) == 103
assert count_distinct_results(5) == 10           # {0..9}: ánh xạ 5->b cho b = 0..9; không đổi cho ra 5
assert count_distinct_results(11) == 10          # 0, 11, 22, ..., 99
assert count_distinct_results(12) == 19          # {2,12,...,92} và {10..19} chung đúng giá trị 12
random.seed(66)
for _ in range(300):
    n = gen()
    assert max_after_remap(n) == max(int(str(n).replace(a, b)) for a in "0123456789" for b in "0123456789")
    assert min_no_leading_zero(n) >= 1
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tham lam từ trái sang phải; brute 100 ánh xạ để kiểm chứng)

```python
class Solution:
    def minMaxDifference(self, num: int) -> int:
        s = str(num)
        big = num
        for ch in s:
            if ch != "9":
                big = int(s.replace(ch, "9"))
                break
        small = int(s.replace(s[0], "0"))
        return big - small
```

**Checklist**

- [ ] `max`: đổi chữ số **không phải 9 đầu tiên (từ trái)** thành `9`, thay **mọi** lần xuất hiện?
- [ ] `min`: đổi chữ số **đầu** thành `0` (số `0` ở đầu được phép)?
- [ ] Đáp án là `max - min`, và hai giá trị tính **độc lập** từ `num` gốc?
- [ ] Trường hợp mọi chữ số là `9` (không có chữ số để tăng) đã được xử lý?
- [ ] Đã thử `11891 -> 99009` và `90 -> 99`?
- [ ] Dùng `str.replace` không giới hạn `count` (thay mọi lần xuất hiện)?
- [ ] Signature đúng `minMaxDifference(self, num: int) -> int`?